In [68]:
import unittest
import cv2
import numpy as np
import os
from PIL import Image
import pandas as pd
import sqlite3
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity


def extract_basic_metadata(image_folder):
    metadata_list = []

    for file_name in os.listdir(image_folder):
        file_path = os.path.join(image_folder, file_name)

        if is_valid_image(file_path):
           
            try:
                image = Image.open(file_path)

                # Extract Metadata
                file_name = os.path.basename(file_path)
                file_size = os.path.getsize(file_path)
                image_format = image.format
                image_width, image_height = image.size

                # build a Metadaten-Dictionary and add to List 
                metadata = {
                    'file_name': file_name,
                    'file_size': file_size,
                    'image_format': image_format,
                    'image_width': image_width,
                    'image_height': image_height,
                    
                }
                metadata_list.append(metadata)

                
                image.close()

            except IOError:
                print(f"load failed: {file_path}")

    return metadata_list


def create_database(database_path):
    conn = sqlite3.connect(database_path)
    c = conn.cursor()

    c.execute('''CREATE TABLE IF NOT EXISTS images
                 (id INTEGER PRIMARY KEY AUTOINCREMENT, file_name TEXT, location TEXT, file_path TEXT)''')

    conn.commit()
    conn.close()
    
    

    
def is_valid_image(file_path):
    supported_extensions = ['.jpg', '.jpeg', '.png', '.gif']
    file_extension = os.path.splitext(file_path)[1].lower()
    return file_extension in supported_extensions

def insert_image_data(database_path, file_name, location, file_path):
    conn = sqlite3.connect(database_path)
    c = conn.cursor()

    c.execute("INSERT INTO images (file_name, location, file_path) VALUES (?, ?, ?)", (file_name, location, file_path))

    conn.commit()
    conn.close()

def cosine_similarity(vector1, vector2):
    dot_product = np.dot(vector1, vector2)
    norm1 = np.linalg.norm(vector1)
    norm2 = np.linalg.norm(vector2)
    similarity = dot_product / (norm1 * norm2)
    return similarity
    

class ImageSimilarityTests(unittest.TestCase):
    def test_calculate_color_profile_similarity(self):
        image1_path = '/Users/shadobanbouk/Desktop/testbilderr/1868.jpg'
        image2_path = '/Users/shadobanbouk/Desktop/testbilderr/1887.jpg'
        image1 = cv2.imread(image1_path)
        image2 = cv2.imread(image2_path)
        
        # Calculate color profiles for the images
        hist1 = calculate_color_profile(image1)
        hist2 = calculate_color_profile(image2)
        
        # Calculate similarity using the function
        similarity = calculate_color_profile_similarity(hist1, hist2)
        
        # Compare with expected similarity
        expected_similarity = 1 - cv2.compareHist(hist1, hist2, cv2.HISTCMP_BHATTACHARYYA)
        self.assertAlmostEqual(similarity, expected_similarity)
        

    def test_cosine_similarity(self):
        # Vektors forr the Cosine Similarity calculation
        vector1 = np.array([0.2, 0.4, 0.6])
        vector2 = np.array([0.3, 0.5, 0.7])
        
    
        similarity = cosine_similarity(vector1, vector2)
        
        # expected Similarity 
        expected_similarity = 0.9974149030430577
        
        self.assertAlmostEqual(similarity, expected_similarity)

    
    def test_compare_color_distributions(self):
        # Create sample color distributions
        input_distribution = np.array([0.1, 0.2, 0.3, 0.4])
        color_data = pd.DataFrame({
            'Image ID': ['1940.jpg', '1941.jpg', '1942.jpg'],
            'File Path': ['/Users/shadobanbouk/Desktop/testbilderr/1940.jpg',
                          '/Users/shadobanbouk/Desktop/testbilderr/1941.jpg',
                          '/Users/shadobanbouk/Desktop/testbilderr/1942.jpg'],
            'Color Distribution': [np.array([0.2, 0.3, 0.4, 0.5]),
                                   np.array([0.3, 0.4, 0.5, 0.6]),
                                   np.array([0.4, 0.5, 0.6, 0.7])]
        })
        
        # Compare distributions using the function
        similar_images = compare_color_distributions(input_distribution, color_data, top_n=2)
        
        self.assertEqual(len(similar_images), 2)
        



    def test_extract_basic_metadata(self):
        # Create a temporary image folder with sample images
        image_folder = '/Users/shadobanbouk/Desktop/testbilderr'

        metadata_list = extract_basic_metadata(image_folder)

        # Check if the metadata list contains the correct number of entries
        expected_num_images = len(os.listdir(image_folder))
        self.assertEqual(len(metadata_list), expected_num_images)

        # Check the metadata for each image
        for metadata in metadata_list:
            file_name = metadata['file_name']
            file_path = os.path.join(image_folder, file_name)
            image = Image.open(file_path)
            image_format = image.format
            image_width, image_height = image.size

            self.assertEqual(metadata['file_size'], os.path.getsize(file_path))
            self.assertEqual(metadata['image_format'], image_format)
            self.assertEqual(metadata['image_width'], image_width)
            self.assertEqual(metadata['image_height'], image_height)

            
            image.close()

    
    def test_insert_image_data(self):
        # Create a temporary test database
        database_path = '/Users/shadobanbouk/Desktop/test_database.db'
        create_database(database_path)
        
        # Insert sample image data into the database
        file_name = '1841.jpg'
        location = '/Users/shadobanbouk/Desktop/testbilderr'
        file_path = '/Users/shadobanbouk/Desktop/testbilderr/1841.jpg'
        insert_image_data(database_path, file_name, location, file_path)
        
        # Retrieve the inserted image data from the database
        conn = sqlite3.connect(database_path)
        c = conn.cursor()
        c.execute("SELECT * FROM images")
        image_data = c.fetchall()
        conn.close()
        
        # Check if the image data in the database matches the inserted data
        self.assertEqual(len(image_data), 1)
        self.assertEqual(image_data[0][1], file_name)
        self.assertEqual(image_data[0][2], location)
        self.assertEqual(image_data[0][3], file_path)
        
        os.remove(database_path)

if __name__ == '__main__':
    unittest.main(argv=[''], exit=False)


.....
----------------------------------------------------------------------
Ran 5 tests in 0.150s

OK
